In [ ]:
import kagglehub
kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [ ]:
import sys, os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

sys.path.insert(0, os.path.abspath('..'))

from src import utils
from src.utils import load_metabric_data, SmartClinicalImputer, DiscordantSignatureAdder

df = load_metabric_data()
df_LumA = df[df['pam50_+_claudin-low_subtype'] == 'LumA'].copy()

Fetching METABRIC dataset from Kaggle...


100%|██████████| 2.72M/2.72M [00:01<00:00, 1.78MB/s]

Extracting files...


In [5]:
df_LumA['target_mortality'] = df_LumA['death_from_cancer'].apply(
    lambda x: 1 if str(x).strip().lower() == 'died of disease' else 0
)

# Drop leakage
leakage_cols = ['patient_id', 'overall_survival_months', 'overall_survival',
                'death_from_cancer', 'target_mortality', 'chemotherapy',
                'hormone_therapy', 'radio_therapy', 'type_of_breast_surgery']
X_raw = df_LumA.drop(columns=[col for col in leakage_cols if col in df_LumA.columns])
y = df_LumA['target_mortality']

# Encoding
for col in ["er_status", "her2_status", "pr_status"]:
    if col in X_raw.columns: X_raw[col] = (X_raw[col] == "Positive").astype(int)
if "cellularity" in X_raw.columns:
    X_raw["cellularity"] = X_raw["cellularity"].map({"Low": 0, "Moderate": 1, "High": 2})

mutation_cols = [c for c in X_raw.columns if c.endswith("_mut")]
for col in mutation_cols:
    X_raw[col] = X_raw[col].apply(lambda x: 0 if pd.isna(x) or str(x).strip() == "0" else 1)

X = X_raw.select_dtypes(include=['number'])

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Define the Pipeline
preprocessing_steps = Pipeline(steps=[
    ("molecular_signature", DiscordantSignatureAdder()),
    ("smart_imputer", SmartClinicalImputer()),
    ("median_imputation", SimpleImputer(strategy="median")),
    ("scaling", StandardScaler())
])

# Define the ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[("numeric_transform", preprocessing_steps, X.columns.tolist())]
)

# Execute Fit & Transform
X_train_transformed_raw = preprocessor.fit_transform(X_train)
X_test_transformed_raw = preprocessor.transform(X_test)

# Map features back to a DataFrame
column_names = X.columns.tolist() + ['discordant_molecular_score']

X_train_transformed = pd.DataFrame(
    X_train_transformed_raw,
    columns=column_names,
    index=X_train.index
)

X_test_transformed = pd.DataFrame(
    X_test_transformed_raw,
    columns=column_names,
    index=X_test.index
)

Preprocessing pipeline complete. Final shape: (543, 675)


In [12]:
print(f"Final shape: {X_train_transformed.shape}")
X_train_transformed.head()

Final shape: (543, 675)


,age_at_diagnosis,cellularity,cohort,er_status,neoplasm_histologic_grade,her2_status,lymph_nodes_examined_positive,mutation_count,nottingham_prognostic_index,pr_status,...,ppp2cb_mut,smarcd1_mut,nras_mut,ndfip1_mut,hras_mut,prps2_mut,smarcb1_mut,stmn2_mut,siah1_mut,discordant_molecular_score
1003,1.233557,0.964813,0.362207,0.074536,-0.057436,-0.190419,-0.483154,-0.178616,-0.520578,-1.838214,...,0.0,-0.042954,-0.042954,0.0,-0.042954,-0.042954,0.0,-0.042954,0.0,0.152029
1360,1.833490,0.964813,0.362207,0.074536,-0.057436,-0.190419,-0.483154,-0.514217,-0.524252,-1.838214,...,0.0,-0.042954,-0.042954,0.0,-0.042954,-0.042954,0.0,-0.042954,0.0,1.044257
1,-1.594128,0.964813,-1.348040,0.074536,1.427701,-0.190419,-0.483154,-1.185419,0.375805,0.544006,...,0.0,-0.042954,-0.042954,0.0,-0.042954,-0.042954,0.0,-0.042954,0.0,0.573382
1072,-0.494251,-0.622743,0.362207,0.074536,-0.057436,-0.190419,-0.483154,1.499388,-0.515068,0.544006,...,0.0,-0.042954,-0.042954,0.0,-0.042954,-0.042954,0.0,-0.042954,0.0,1.073288
1641,1.763898,-0.622743,2.072454,0.074536,-0.057436,-0.190419,-0.483154,-0.514217,-0.524252,0.544006,...,0.0,-0.042954,-0.042954,0.0,-0.042954,-0.042954,0.0,-0.042954,0.0,-0.299999
